# KAN on embeddings (ResNet18 backbone frozen)

Entrenamiento eficiente en CPU: extrae embeddings con ResNet18 y entrena KAN sobre esos embeddings.
Si, esto permite extraer splines (sobre dimensiones del embedding).
La interpretabilidad es respecto a canales del backbone, no pixeles directos.


In [26]:
# %pip install pykan
import importlib.util
print('KAN available:', importlib.util.find_spec('kan') is not None)


KAN available: True


In [27]:
from pathlib import Path
from collections import Counter
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE_BACKBONE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_KAN = torch.device('cpu')
print('Backbone device:', DEVICE_BACKBONE, '| KAN device:', DEVICE_KAN)


Backbone device: cuda | KAN device: cpu


In [28]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
TRAIN_CSV = ROOT / 'src/data/processed/manifest_train.csv'
TEST_CSV = ROOT / 'src/data/processed/manifest_test.csv'
TUNED_BACKBONE = ROOT / 'reports/models/resnet18_tuned_best.pt'
KAN_HEAD_PATH = ROOT / 'reports/models/kan_head_embeddings.pt'

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
patients = train_df['patient_id'].dropna().unique()
train_pat, val_pat = train_test_split(patients, test_size=0.2, random_state=SEED)
tr_df = train_df[train_df['patient_id'].isin(train_pat)].copy()
val_df = train_df[train_df['patient_id'].isin(val_pat)].copy()
print('Train:', tr_df.shape)
print('Val:', val_df.shape)
print('Test:', test_df.shape)
print('Tuned backbone exists:', TUNED_BACKBONE.exists())


Train: (2315, 10)
Val: (549, 10)
Test: (422, 10)
Tuned backbone exists: True


In [29]:
class MammographyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        x = Image.open(r['image_path_local']).convert('RGB')
        x = self.transform(x)
        y = int(r['label'])
        return x, y

tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_loader = DataLoader(MammographyDataset(tr_df, tfm), batch_size=32, shuffle=False, num_workers=0)
val_loader = DataLoader(MammographyDataset(val_df, tfm), batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(MammographyDataset(test_df, tfm), batch_size=32, shuffle=False, num_workers=0)


In [30]:
class ResNet18Backbone(nn.Module):
    def __init__(self, tuned_path: Path):
        super().__init__()
        full = models.resnet18(weights=None)
        full.fc = nn.Linear(full.fc.in_features, 2)
        full.load_state_dict(torch.load(tuned_path, map_location=DEVICE_BACKBONE))
        self.features = nn.Sequential(*list(full.children())[:-1])
    def forward(self, x):
        z = self.features(x).flatten(1)
        return z

backbone = ResNet18Backbone(TUNED_BACKBONE).to(DEVICE_BACKBONE).eval()
for p in backbone.parameters():
    p.requires_grad = False
backbone


ResNet18Backbone(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_s

In [31]:
def extract_embeddings(loader):
    Xs, ys = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE_BACKBONE)
            z = backbone(x)
            Xs.append(z.cpu())
            ys.append(y.cpu())
    return torch.cat(Xs, 0), torch.cat(ys, 0)

Xtr, ytr = extract_embeddings(train_loader)
Xva, yva = extract_embeddings(val_loader)
Xte, yte = extract_embeddings(test_loader)

mu = Xtr.mean(0, keepdim=True)
sigma = Xtr.std(0, keepdim=True).clamp_min(1e-6)
Xtr = (Xtr - mu) / sigma
Xva = (Xva - mu) / sigma
Xte = (Xte - mu) / sigma

Xtr_h, ytr_h = Xtr.to(DEVICE_KAN), ytr.to(DEVICE_KAN)
Xva_h, yva_h = Xva.to(DEVICE_KAN), yva.to(DEVICE_KAN)
Xte_h, yte_h = Xte.to(DEVICE_KAN), yte.to(DEVICE_KAN)


In [32]:
if not importlib.util.find_spec("kan"):
    raise RuntimeError("KAN no instalado. Ejecuta `%pip install pykan`, reinicia kernel y corre de nuevo.")
from kan import KAN

counts = Counter(ytr.tolist())
w = torch.tensor([len(ytr)/(2*counts[0]), len(ytr)/(2*counts[1])], dtype=torch.float32, device=DEVICE_KAN)
criterion = nn.CrossEntropyLoss(weight=w)

kan = KAN(width=[512,32,2], grid=5, k=3).to(DEVICE_KAN)
opt = torch.optim.Adam(kan.parameters(), lr=5e-4)
sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)

best_auc, best_state, wait, patience = -1.0, None, 0, 8
for ep in range(1, 41):
    kan.train(); opt.zero_grad()
    logits = kan(Xtr_h)
    loss = criterion(logits, ytr_h)
    loss.backward(); opt.step()
    kan.eval()
    with torch.no_grad():
        val_logits = kan(Xva_h)
        prob = torch.softmax(val_logits, dim=1)[:,1].cpu().numpy()
        y_true = yva_h.cpu().numpy()
        auc = np.nan if len(np.unique(y_true)) < 2 else roc_auc_score(y_true, prob)
    sch.step(auc if not np.isnan(auc) else 0.0)
    print(f"KAN E{ep:02d} loss={loss.item():.4f} val_auc={auc:.4f}")
    if auc > best_auc:
        best_auc, wait = auc, 0
        best_state = {k:v.cpu().clone() for k,v in kan.state_dict().items()}
    else:
        wait += 1
        if wait >= patience:
            print("Early stop KAN"); break

kan.load_state_dict(best_state)
kan.eval()
torch.save(kan.state_dict(), KAN_HEAD_PATH)
print("Saved:", KAN_HEAD_PATH)


checkpoint directory created: ./model
saving model version 0.0
KAN E01 loss=0.6965 val_auc=0.5387
KAN E02 loss=0.6741 val_auc=0.5917
KAN E03 loss=0.6525 val_auc=0.6304
KAN E04 loss=0.6316 val_auc=0.6572
KAN E05 loss=0.6115 val_auc=0.6734
KAN E06 loss=0.5919 val_auc=0.6834
KAN E07 loss=0.5729 val_auc=0.6906
KAN E08 loss=0.5543 val_auc=0.6963
KAN E09 loss=0.5362 val_auc=0.6993
KAN E10 loss=0.5185 val_auc=0.7029
KAN E11 loss=0.5013 val_auc=0.7042
KAN E12 loss=0.4844 val_auc=0.7057
KAN E13 loss=0.4681 val_auc=0.7076
KAN E14 loss=0.4523 val_auc=0.7086
KAN E15 loss=0.4370 val_auc=0.7099
KAN E16 loss=0.4223 val_auc=0.7107
KAN E17 loss=0.4083 val_auc=0.7119
KAN E18 loss=0.3948 val_auc=0.7130
KAN E19 loss=0.3821 val_auc=0.7136
KAN E20 loss=0.3700 val_auc=0.7144
KAN E21 loss=0.3586 val_auc=0.7150
KAN E22 loss=0.3479 val_auc=0.7156
KAN E23 loss=0.3379 val_auc=0.7160
KAN E24 loss=0.3284 val_auc=0.7167
KAN E25 loss=0.3196 val_auc=0.7170
KAN E26 loss=0.3114 val_auc=0.7173
KAN E27 loss=0.3037 val_auc